In [1]:
from pathlib import Path
import json
import re
import unicodedata

In [2]:
EXCERPT_ID = "excerpt_001"

PROJECT_ROOT = Path("/workspace/storyfx")

SEGMENT_SPEC_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "composition_plans"
    / f"pred_{EXCERPT_ID}_segment_specification.json"
)

FA_PARSED_PATH = (
    PROJECT_ROOT
    / "data"
    / EXCERPT_ID
    / "alignment"
    / f"{EXCERPT_ID}_fa_parsed.txt"
)

OUTPUT_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "schedules"
    / f"{EXCERPT_ID}_segment_schedule.json"
)


print("Segment specification:", SEGMENT_SPEC_PATH)
print("Forced alignment:", FA_PARSED_PATH)
print("Output:", OUTPUT_PATH)

Segment specification: /workspace/storyfx/outputs/composition_plans/pred_excerpt_001_segment_specification.json
Forced alignment: /workspace/storyfx/data/excerpt_001/alignment/excerpt_001_fa_parsed.txt
Output: /workspace/storyfx/outputs/schedules/excerpt_001_segment_schedule.json


In [3]:
def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)

    text = (
        text
        .replace("’", "'")
        .replace("‘", "'")
        .replace("“", '"')
        .replace("”", '"')
    )

    return text.lower()

In [4]:
def lexical_tokens(text):
    """
    Convert text into lexical tokens.

    Examples:
        "rosy-coloured" -> ["rosy", "coloured"]
        "don't" -> ["don't"]
        "room," -> ["room"]
    """
    text = normalize_text(text)

    return re.findall(
        r"[a-z0-9]+(?:'[a-z0-9]+)?",
        text,
    )

In [5]:
def flatten_fa_words(fa_words):
    flat_tokens = []
    word_indices = []

    for i, word in enumerate(fa_words):
        tokens = lexical_tokens(word["word"])

        for token in tokens:
            flat_tokens.append(token)
            word_indices.append(i)

    return flat_tokens, word_indices

In [6]:
def load_fa_words(path):
    text = Path(path).read_text(
        encoding="utf-8"
    ).strip()

    words = []

    for item in text.split():
        if "|" not in item:
            continue

        word, end_time = item.rsplit("|", 1)

        words.append(
            {
                "word": word,
                "normalized": normalize_text(word),
                "end_time": float(end_time),
            }
        )

    # Infer each word's start time from the preceding word's end.
    for i, word in enumerate(words):
        if i == 0:
            word["start_time"] = 0.0
        else:
            word["start_time"] = words[i - 1]["end_time"]

    return words

In [7]:
fa_words = load_fa_words(
    FA_PARSED_PATH
)

print("Aligned words:", len(fa_words))
print(fa_words[:10])

Aligned words: 971
[{'word': 'for', 'normalized': 'for', 'end_time': 0.2, 'start_time': 0.0}, {'word': 'three', 'normalized': 'three', 'end_time': 0.54, 'start_time': 0.2}, {'word': 'centuries', 'normalized': 'centuries', 'end_time': 1.47, 'start_time': 0.54}, {'word': "harrow's", 'normalized': "harrow's", 'end_time': 2.03, 'start_time': 1.47}, {'word': 'peak', 'normalized': 'peak', 'end_time': 2.4, 'start_time': 2.03}, {'word': 'had', 'normalized': 'had', 'end_time': 2.55, 'start_time': 2.4}, {'word': 'stood', 'normalized': 'stood', 'end_time': 2.83, 'start_time': 2.55}, {'word': 'as', 'normalized': 'as', 'end_time': 2.96, 'start_time': 2.83}, {'word': 'a', 'normalized': 'a', 'end_time': 3.03, 'start_time': 2.96}, {'word': 'testament', 'normalized': 'testament', 'end_time': 3.71, 'start_time': 3.03}]


In [8]:
def anchor_tokens(anchor):
    tokens = []

    for word in anchor.split():
        normalized = normalize_word(word)

        if normalized:
            tokens.append(normalized)

    return tokens

In [9]:
def find_anchor(
    anchor,
    fa_words,
    start_index=0,
):
    target = lexical_tokens(anchor)

    if not target:
        raise ValueError(
            f"Anchor contains no matchable words: {anchor}"
        )

    flat_tokens, word_indices = flatten_fa_words(
        fa_words
    )

    # Convert the MFA word-level start index
    # into a flattened-token start position.
    flat_start = 0

    while (
        flat_start < len(word_indices)
        and word_indices[flat_start] < start_index
    ):
        flat_start += 1

    target_length = len(target)

    for i in range(
        flat_start,
        len(flat_tokens) - target_length + 1,
    ):
        candidate = flat_tokens[
            i:i + target_length
        ]

        if candidate == target:
            first_word_index = word_indices[i]

            last_word_index = word_indices[
                i + target_length - 1
            ]

            return (
                first_word_index,
                last_word_index,
            )

    raise ValueError(
        "Could not match anchor:\n"
        f"{anchor}\n\n"
        f"Normalized tokens:\n{target}"
    )

In [10]:
def quantify_segments(
    segment_spec_path,
    fa_parsed_path,
):
    segments = json.loads(
        Path(segment_spec_path).read_text(
            encoding="utf-8"
        )
    )

    fa_words = load_fa_words(
        fa_parsed_path
    )

    scheduled_segments = []

    search_index = 0

    for segment in segments:
        start_anchor = segment["start"]
        end_anchor = segment["end"]

        # --------------------------------------------
        # Match start anchor
        # --------------------------------------------

        start_first, start_last = find_anchor(
            start_anchor,
            fa_words,
            start_index=search_index,
        )

        # --------------------------------------------
        # Match end anchor
        # --------------------------------------------

        end_first, end_last = find_anchor(
            end_anchor,
            fa_words,
            start_index=start_first,
        )

        if end_last < start_first:
            raise ValueError(
                f"End anchor occurs before start anchor "
                f"for {segment['segment_id']}"
            )

        start_time = fa_words[
            start_first
        ]["start_time"]

        end_time = fa_words[
            end_last
        ]["end_time"]

        quantified = {
            **segment,
            "start_time": round(
                start_time,
                2,
            ),
            "end_time": round(
                end_time,
                2,
            ),
        }

        scheduled_segments.append(
            quantified
        )

        # Future segments must occur after this one.
        search_index = end_last + 1

    return scheduled_segments

In [11]:
segment_schedule = quantify_segments(
    SEGMENT_SPEC_PATH,
    FA_PARSED_PATH,
)

segment_schedule

[{'segment_id': 'segment_000',
  'segment_type': 'expository',
  'start': 'For three centuries, Harrow’s Peak',
  'end': 'or rather, dare not disobey.',
  'reverb_profile': None,
  'start_time': 0.0,
  'end_time': 91.6},
 {'segment_id': 'segment_001',
  'segment_type': 'dramatized',
  'start': 'That night, Corbin barely heard',
  'end': 'burning lantern off the dresser.',
  'reverb_profile': 'indoor_medium_absorbed_regular',
  'start_time': 91.6,
  'end_time': 107.11},
 {'segment_id': 'segment_002',
  'segment_type': 'dramatized',
  'start': 'From his quarters, his anxious',
  'end': 'black legacy by that night.',
  'reverb_profile': 'outdoor_open',
  'start_time': 107.11,
  'end_time': 159.9},
 {'segment_id': 'segment_003',
  'segment_type': 'dramatized',
  'start': 'Pushing open the massive oak',
  'end': 'did not materialize in reality.',
  'reverb_profile': 'indoor_large_reflective',
  'start_time': 159.9,
  'end_time': 226.25},
 {'segment_id': 'segment_004',
  'segment_type': 'dra

In [12]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)
1
OUTPUT_PATH.write_text(
    json.dumps(
        segment_schedule,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(
    f"Segment schedule saved to: {OUTPUT_PATH}"
)

Segment schedule saved to: /workspace/storyfx/outputs/schedules/excerpt_001_segment_schedule.json
